In [ ]:
!pip install -q -U agno openai duckduckgo-search ddgs

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

Enter OPENAI_API_KEY: ··········


In [ ]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.duckduckgo import DuckDuckGoTools

In [ ]:
TOPIC = "Agentic AI in Enterprise Automation"
TARGET_AUDIENCE = "CTOs and technology leaders"
TONE = "authoritative and inspiring"

In [ ]:
# 1) Planner: creates a simple plan/checklist
planner = Agent(
    name="Planner",
    model=OpenAIChat(id="gpt-4o-mini"),
    markdown=True,
    instructions=[
        "Create a step-by-step plan to produce the executive brief.",
        "Output ONLY a numbered plan of 5-8 steps.",
        "Include a checklist of the strict rules to verify at the end."
    ],
)

# 2) Researcher: MUST use web tool, returns facts + sources
researcher = Agent(
    name="Researcher",
    model=OpenAIChat(id="gpt-4o-mini"),
    tools=[DuckDuckGoTools()],
    markdown=True,
    instructions=[
        "You are an enterprise tech researcher.",
        "Use web_search for factual claims.",
        "Collect: 5 numeric facts + 4 trends + 2-4 risk references + 1 expert quote candidate.",
        "Output MUST be in this format:",
        "## Facts (5 bullets) -> each bullet includes a number and a source link",
        "## Trends (4 bullets) -> each bullet includes at least 1 source link",
        "## Risks refs (2-4 bullets) -> each bullet includes a source link",
        "## Quote candidates (1-2 bullets) -> each bullet includes person, org, quote, link",
        "## Source Links (5-10 links)"
    ],
)

# 3) Writer: writes the final brief ONLY from research text
writer = Agent(
    name="Writer",
    model=OpenAIChat(id="gpt-4o-mini"),
    markdown=True,
    instructions=[
        f"You write concise executive briefs for {TARGET_AUDIENCE}.",
        f"Tone must be {TONE}.",
        "You will be given research notes with links. Use ONLY those facts/links.",
        "STRICT RULES:",
        "1) Provide exactly 5 key facts, and EACH fact MUST include at least one number and end with an inline link.",
        "2) Provide exactly 4 trends. Each trend must have at least one inline link.",
        "3) Provide enterprise risks + mitigations (security, compliance, reliability, privacy) and include at least 2 inline links in this section.",
        "4) Provide a 30/60/90 day roadmap with bullets.",
        "5) Provide exactly 1 expert quote with attribution (Person + Organization) and a source link.",
        "6) Keep total length ~500–700 words.",
        "7) Use these exact headings:",
        "## Executive Summary (3–4 lines)",
        "## 1) Key Facts (5 bullets, each with a number + inline citation link)",
        "## 2) Current Trends (4 bullets, each with at least 1 citation link)",
        "## 3) Enterprise Risks & Mitigations (bullets; include at least 2 citations in this section)",
        "## 4) 30/60/90-Day Roadmap (bullets)",
        "## 5) Expert Quote (exactly 1 quote with attribution + link)",
        "## 6) Source Links (5–8 links)",
    ],
)

# 4) Reviewer: checks compliance; returns PASS/FAIL + fixes
reviewer = Agent(
    name="Reviewer",
    model=OpenAIChat(id="gpt-4o-mini"),
    markdown=True,
    instructions=[
        "You are a strict compliance reviewer for the brief.",
        "Check the output against ALL rules.",
        "Return ONLY in this format:",
        "STATUS: PASS or FAIL",
        "ISSUES: bullet list (if FAIL)",
        "FIX_INSTRUCTIONS: bullet list of exact edits needed (if FAIL)",
        "Do not rewrite the full brief."
    ],
)

In [ ]:
def run_agentic_brief(topic: str, max_rounds: int = 2):
    # Step A: Plan
    plan = planner.run(f"Create a plan to write an executive tech brief on: {topic}")
    print("\n================ PLAN ================\n")
    print(plan.content)

    # Step B: Research
    research = researcher.run(f"Research for an executive tech brief on: {topic}")
    print("\n================ RESEARCH ================\n")
    print(research.content)

    # Step C: Write first draft
    draft = writer.run(f"""
Topic: {topic}

Here are the research notes (use ONLY these facts and links):
{research.content}
""")

    # Step D: Review + Fix loop
    for round_no in range(1, max_rounds + 1):
        print(f"\n================ DRAFT (Round {round_no}) ================\n")
        print(draft.content)

        review = reviewer.run(draft.content)
        print(f"\n================ REVIEW (Round {round_no}) ================\n")
        print(review.content)

        if "STATUS: PASS" in review.content:
            return draft.content  # final

        # If FAIL -> ask writer to apply fixes
        draft = writer.run(f"""
You must fix the brief using the reviewer instructions.

Reviewer feedback:
{review.content}

Original brief:
{draft.content}
""")

    # return best effort even if still failing
    return draft.content

final_brief = run_agentic_brief(TOPIC, max_rounds=2)

print("\n================ FINAL BRIEF ================\n")
print(final_brief)


================ PLAN ================

# Step-by-Step Plan to Produce the Executive Brief on Agentic AI in Enterprise Automation

1. **Define the Scope and Purpose**
   - Identify the key objectives of the brief.
   - Outline target audience and their needs.

2. **Conduct Preliminary Research**
   - Gather information on the current state of Agentic AI in enterprise automation.
   - Identify relevant case studies and examples.
   - Explore potential benefits and challenges.

3. **Develop an Outline**
   - Create a structured outline for the brief, including sections for introduction, definitions, benefits, case studies, challenges, and conclusion.

4. **Write Draft Content**
   - Begin populating the outline with detailed content.
   - Ensure clarity and conciseness while conveying technical information effectively.

5. **Solicit Feedback**
   - Share the draft with colleagues or experts for their input and suggestions.
   - Incorporate feedback to enhance the quality of the brief.

